# 03 - Task 2: Content-Based Restaurant Recommender

Build recommendations using cuisine text similarity plus numeric context (price and rating).

Leakage policy is defined in Notebook 01. This recommender does not use target leakage from Task 1 modeling.

In [8]:
# Shared setup
import warnings
import random
import numpy as np
import pandas as pd

from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
warnings.filterwarnings('ignore')

print('pandas:', pd.__version__)
print('numpy:', np.__version__)
print('seed:', SEED)

pandas: 2.2.3
numpy: 2.0.2
seed: 42


In [9]:
DATA_PATH = 'Dataset .csv'
df_raw = pd.read_csv(DATA_PATH)

EXPECTED_COLUMNS = [
    'Restaurant ID', 'Restaurant Name', 'Country Code', 'City', 'Address',
    'Locality', 'Locality Verbose', 'Longitude', 'Latitude', 'Cuisines',
    'Average Cost for two', 'Currency', 'Has Table booking',
    'Has Online delivery', 'Is delivering now', 'Switch to order menu',
    'Price range', 'Aggregate rating', 'Rating color', 'Rating text', 'Votes'
]
missing = sorted(set(EXPECTED_COLUMNS) - set(df_raw.columns))
assert not missing, f'Missing required columns: {missing}'
print('Loaded shape:', df_raw.shape)

Loaded shape: (9551, 21)


In [10]:
UNIVERSAL_DROP_COLUMNS = [
    'Restaurant ID', 'Address', 'Locality', 'Locality Verbose',
    'Longitude', 'Latitude', 'Switch to order menu',
    'Rating color', 'Rating text'
 ]
YES_NO_COLUMNS = ['Has Table booking', 'Has Online delivery', 'Is delivering now']

def map_yes_no(series: pd.Series) -> pd.Series:
    mapping = {'Yes': 1, 'No': 0}
    return series.map(mapping).fillna(series)

def cuisine_tokens(value):
    if pd.isna(value):
        return ['Unknown']
    tokens = [x.strip() for x in str(value).split(',') if x.strip()]
    return tokens if tokens else ['Unknown']

def apply_shared_preprocessing(df: pd.DataFrame, task: str) -> pd.DataFrame:
    out = df.copy()

    for c in YES_NO_COLUMNS:
        if c in out.columns:
            out[c] = map_yes_no(out[c])

    if task in {'task1', 'task2'}:
        out['Cuisines'] = out['Cuisines'].fillna('Unknown')
    elif task == 'task3':
        out = out.dropna(subset=['Cuisines']).copy()

    out['Cuisine Tokens'] = out['Cuisines'].apply(cuisine_tokens)
    out['Cuisine Text'] = out['Cuisine Tokens'].apply(lambda xs: ' '.join(xs).lower())

    drop_cols = [c for c in UNIVERSAL_DROP_COLUMNS if c in out.columns]
    out = out.drop(columns=drop_cols)
    return out

In [11]:
df = apply_shared_preprocessing(df_raw, task='task2')

# Anti-leakage guardrail for this task: do not use rating text/color semantics in content matrix.
forbidden_cols = ['Rating color', 'Rating text']
assert not any(c in df.columns for c in forbidden_cols), 'Leakage columns still present.'

vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
cuisine_matrix = vectorizer.fit_transform(df['Cuisine Text'])

num_features = df[['Price range', 'Aggregate rating']].fillna(0).values
num_scaled = MinMaxScaler().fit_transform(num_features)

content_matrix = hstack([cuisine_matrix, num_scaled])
similarity_matrix = cosine_similarity(content_matrix, dense_output=False)

name_to_indices = df.groupby('Restaurant Name').apply(lambda x: list(x.index)).to_dict()
print('Similarity matrix shape:', similarity_matrix.shape)
print('Unique names:', len(name_to_indices))

Similarity matrix shape: (9551, 9551)
Unique names: 7446


In [12]:
def resolve_index(name: str) -> int:
    idxs = name_to_indices.get(name)
    if not idxs:
        raise ValueError(f'Restaurant not found: {name}')
    if len(idxs) == 1:
        return idxs[0]
    # Duplicate names: prefer highest-voted entry.
    candidates = df.loc[idxs].copy()
    return int(candidates.sort_values('Votes', ascending=False).index[0])

def recommend(name: str, top_n: int = 5) -> pd.DataFrame:
    idx = resolve_index(name)
    sim_scores = similarity_matrix[idx].toarray().ravel()

    candidates = df.copy()
    candidates['similarity'] = sim_scores

    candidates = candidates[candidates.index != idx]  # self exclusion
    candidates = candidates[candidates['Aggregate rating'] > 0]  # exclude unrated

    ranked = candidates.sort_values(['similarity', 'Votes'], ascending=[False, False])
    cols = ['Restaurant Name', 'City', 'Cuisines', 'Price range', 'Aggregate rating', 'Votes', 'similarity']
    return ranked[cols].head(top_n).reset_index(drop=True)

In [13]:
# Example 1: high-vote known restaurant
known_name = df.sort_values('Votes', ascending=False)['Restaurant Name'].iloc[0]
print('Known example:', known_name)
display(recommend(known_name, top_n=5))

Known example: Toit


,Restaurant Name,City,Cuisines,Price range,Aggregate rating,Votes,similarity
0,Vintage Machine,Lucknow,"Italian, American, Lebanese",3,4.3,887,0.836922
1,AMPM Caf�� & Bar,New Delhi,"Continental, Italian, American",3,4.1,1980,0.827163
2,Portneuf Valley Brewing,Pocatello,"American, Pizza",2,3.7,191,0.820805
3,Raglan Road Irish Pub and Restaurant,Orlando,Irish,4,4.3,782,0.812483
4,Jon's head Grill,New Delhi,"Mexican, Italian, American",3,3.9,105,0.801221


In [14]:
# Example 2: rare-cuisine restaurant
exploded = df[['Restaurant Name', 'Cuisine Tokens']].explode('Cuisine Tokens')
counts = exploded['Cuisine Tokens'].value_counts()
rare_cuisines = counts[counts <= 3].index.tolist()

if rare_cuisines:
    rare_name = exploded[exploded['Cuisine Tokens'] == rare_cuisines[0]]['Restaurant Name'].iloc[0]
    print('Rare cuisine example:', rare_name, '| cuisine:', rare_cuisines[0])
    display(recommend(rare_name, top_n=5))
else:
    print('No sufficiently rare cuisine found with threshold <= 3.')

Rare cuisine example: Summer Pavilion | cuisine: Dim Sum


,Restaurant Name,City,Cuisines,Price range,Aggregate rating,Votes,similarity
0,Yauatcha,London,"Chinese, Dim Sum",4,4.7,1326,0.876967
1,Hakkasan,London,"Chinese, Dim Sum",4,4.8,395,0.876904
2,Raglan Road Irish Pub and Restaurant,Orlando,Irish,4,4.3,782,0.786667
3,Woks - The Lalit New Delhi,New Delhi,"Chinese, Seafood",4,3.6,62,0.786166
4,Rice Bowl,New Delhi,"Chinese, Seafood",3,3.7,241,0.745039


In [15]:
# Example 3: low-rated restaurant
low_df = df[(df['Aggregate rating'] > 0) & (df['Aggregate rating'] <= 2.5)]
if not low_df.empty:
    low_name = low_df.sort_values('Aggregate rating', ascending=True)['Restaurant Name'].iloc[0]
    print('Low-rated example:', low_name)
    display(recommend(low_name, top_n=5))
else:
    print('No low-rated restaurants available in current dataset filter.')

Low-rated example: Pind Balluchi


,Restaurant Name,City,Cuisines,Price range,Aggregate rating,Votes,similarity
0,Clay 1 Grill,New Delhi,"North Indian, Mughlai",3,3.6,283,1.000000
1,New Minar,New Delhi,"North Indian, Mughlai",3,3.6,229,1.000000
2,Soul Curry - Bellagio,New Delhi,"North Indian, Mughlai",3,3.6,158,1.000000
3,Tandoor Villa,Varanasi,"North Indian, Mughlai",3,3.6,63,1.000000
4,Punjabi By Nature,New Delhi,"North Indian, Mughlai",3,3.7,1256,0.999925
